# 03 — Modelagem

Grupo 1 — Breno Luna e Paula Carlesso

Treino e comparação. O split é **temporal** — os últimos 25% do período viram
teste, para o modelo nunca treinar com transações posteriores às avaliadas.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import ARQUIVO_DATASET, COLUNA_ALVO, FIGURES_DIR

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

In [ ]:
from src.models.train_model import MODELOS, separar_temporal

df = pd.read_parquet(ARQUIVO_DATASET)
X_treino, X_teste, y_treino, y_teste, teste_completo = separar_temporal(df)
X_treino.shape, X_teste.shape

In [ ]:
for nome, modelo in MODELOS.items():
    modelo.fit(X_treino, y_treino)
    print(nome, "treinado")

## Comparação no limiar escolhido por modelo

In [ ]:
from src.models.evaluate import calcular_metricas, escolher_limiar

resultados = []
for nome, modelo in MODELOS.items():
    y_prob = modelo.predict_proba(X_teste)[:, 1]
    limiar = escolher_limiar(y_teste, y_prob, recall_minimo=0.70)
    y_prev = (y_prob >= limiar).astype(int)
    resultados.append({"modelo": nome, "limiar": limiar, **calcular_metricas(y_teste, y_prev, y_prob)})

pd.DataFrame(resultados)[["modelo", "limiar", "recall", "precision", "f1", "auc_pr", "taxa_alerta"]].round(4)

## Importância das features (random forest)

In [ ]:
from src.features.build_features import COLUNAS_MODELO

rf = MODELOS["random_forest"]
importancias = pd.Series(rf.named_steps["clf"].feature_importances_, index=COLUNAS_MODELO)
importancias.sort_values().plot(kind="barh", figsize=(8, 6), title="Importância das features")
plt.tight_layout()